# Build a Debugging Tutor with Gradio

### Learning Objective
In this notebook you will build an AI debugging tutor to learn how system prompts, examples, and settings change the way a language model behaves.

### What You Will Do
- **Edit the system prompt** to change the AI's rules and personality.
- **Toggle few-shot examples** on or off to see how the AI learns from them.
- **Adjust temperature** to control randomness.
- **Try Chain-of-Thought (CoT)** to make the model reason step-by-step before answering.
- **Compare models of different sizes** (optional) to see how scale affects quality.
- **Inspect the White Box** to see the exact tokens sent to the model.

### How to Build Your Own Chatbot
This notebook is a template. To build a different AI assistant, change just three things:
1. `system_prompt` — The rules your assistant follows.
2. `few_shot_examples` — Example conversations that teach the format.
3. `test_cases` — Inputs to test your assistant with.

Everything else (inference, streaming, UI, logging) stays the same.

## 1. Setup

First we import every library the notebook needs. All imports are in this one cell.

**Local inference** means running a model on your own machine. It costs nothing per query but is slower and limited by your hardware.

In [1]:
# %pip -q install gradio llama-cpp-python
import gradio as gr   # Build interactive web UIs
from llama_cpp import Llama  # Run GGUF models locally
import time  # Measure latency
import re    # Parse <thinking> tags

Now we load the model file into memory. This step takes a few seconds.

`context_window_size` sets the maximum number of **tokens** the model can read at once. A token is a small piece of text, roughly 3-4 characters. A bigger window lets the model see more text but uses more memory.

In [2]:
# Change this to match where your model file is located
# model_file_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"
# Qwen2.1 1.5B instruct (fine-tuned) may follow your instruction
# model_file_path = "/home/jovyan/shared/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf" 
# Note: R1 distill model may not follow your instruction.
model_file_path = "/home/jovyan//Small_Models_SP26/Seoha/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"

# Change this to increase or decrease context window
context_window_size = 1024

model = Llama(
    model_path=model_file_path,
    n_ctx=context_window_size,
    n_threads=4,
    verbose=False,
)

print("Local model loaded. Context window:", context_window_size, "tokens.")

llama_context: n_ctx_per_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Local model loaded. Context window: 1024 tokens.


Reflection (Seoha)
You can also observe that small local models sometimes uses Chain-of-Thought (CoT) reasoning. In some cases, the model produces no final answer because it uses up all the output tokens during the thinking phase. When this happens, increasing the maximum token limit (UI model parametes)or reducing the input token length can sometimes allow the final answer to appear. However, even after adjusting the token limits, the model may still fail. This behavior is quite interesting to observe.

## 1b. (Optional) Cloud API Models

Connect to a cloud API to compare your local model with larger ones.

Pick **one** option below (Groq is free, OpenAI is paid). Uncomment the cell, edit `api_models` to add the models you want, then re-run.

The **Model** dropdown in the UI will update automatically with whatever you put in `api_models`.

In [3]:
# These defaults keep the notebook working when no API is enabled.
# The API cells below will override them if uncommented.
api_client = None
api_models = {}

INTERESTING 
- Qwen32B is sometimes smater than Llama70B
- hallucination...process real time...wrong guided...large models....
- look at thinking tab(CoT)

### Option A: Groq ([free](https://console.groq.com/docs/rate-limits)) 
Get a free [keys](https://console.groq.com/keys)

In [4]:
# ── UNCOMMENT THIS CELL TO ENABLE GROQ (free) ──
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(".env")  # Change path if your .env is elsewhere
groq_api_key = os.getenv("GROQ_API_KEY") # create file with .env -> paste the key GROQ_API_KEY=gsk_...
print("Groq API Key:", "loaded" if groq_api_key else "NOT FOUND")

if groq_api_key:
    api_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

    # Add or remove models here. Each line becomes a dropdown option.
    # Format:  "Display Name": "model-id" 
    # Check Full models at Free plan limits: console.groq.com/docs/rate-limits
    # Great for comparing how model size affects CoT quality
    api_models = {
        "llama-3.3-70b": "llama-3.3-70b-versatile",
        "llama-3.1-8b": "llama-3.1-8b-instant",
        "qwen3-32b":    "qwen/qwen3-32b",
    }
    print("Groq models:", list(api_models.keys()))

Groq API Key: loaded
Groq models: ['llama-3.3-70b', 'llama-3.1-8b', 'qwen3-32b']


### Option B: OpenAI (paid)

In [5]:
# ── UNCOMMENT THIS CELL TO ENABLE OPENAI (paid) ──

# from openai import OpenAI
# from dotenv import load_dotenv
# import os
#
# load_dotenv(".env")  # Change path if your .env is elsewhere
# openai_api_key = os.getenv("OPENAI_API_KEY") # .env -> OPENAI_API_KEY=sk-...
# print("OpenAI API Key:", "loaded" if openai_api_key else "NOT FOUND")
#
# if openai_api_key:
#     api_client = OpenAI(api_key=openai_api_key)
#
#     # Add or remove models here. Each line becomes a dropdown option.
#     api_models = {
#         "gpt-4o-mini": "gpt-4o-mini",
#     }
#     print("OpenAI models:", list(api_models.keys()))

## 2. System Prompt

**Prompt engineering** is the skill of writing instructions that control how an LLM behaves.

The system prompt defines three things:
- **Role**: What is the AI pretending to be?
- **Rules**: What must it do? What must it never do?
- **Format**: How should the output look?

Both prompts are editable in the Gradio UI:
- **System Prompt**: The base instructions. Always used.
- **CoT Addition**: Extra instructions appended when CoT is on.

### Experiments to try
- Delete the "NEVER give the corrected code" rule. Does the model start giving answers?
- Edit the CoT addition: change the 4 thinking steps to just 1. Does the output change?
- Change "TA" to "senior engineer". Does the tone change?

In [6]:
# The system prompt defines your assistant's behavior.
# Change this to build a different kind of chatbot.
# Students can also edit this live in the Gradio UI.
system_prompt = """
You are a teaching assistant helping a student debug their code.
Your goal is to help the student discover the fix themselves.

Rules:
- NEVER give the corrected code, the fix, or the direct answer.
- NEVER say what the student should change or type.
- Keep each section under 20 words.
- If the error is ambiguous, ask a clarifying question in Check.

Output format (use these exact headers):

Diagnosis: What symptom the error message describes.
Root cause: Why the code produces that symptom, without revealing the fix.
Check: One step or short example the student can try to verify the cause.
Review: One general principle to prevent this type of bug.
""".strip()

# When CoT is ON, this text is appended after the system prompt.
# Students can also edit this in the Gradio UI.
cot_addition = """
IMPORTANT: Before answering, reason step-by-step inside <thinking> tags.

In your thinking (the student will NOT see this):
1. Read the error. What does it literally say?
2. Read the code. What was the student trying to do?
3. Find the mismatch between intent and code. What exactly is wrong?
4. Plan a hint that points TOWARD the bug WITHOUT saying the fix.
   Ask yourself: if I say this, can the student copy-paste it as the answer?
   If yes, rephrase to be less direct.

After </thinking>, write your answer using the headers above.
""".strip()

print(f" System prompt: {len(system_prompt)} chars, ~{len(system_prompt)//4} tokens")
print(f" CoT addition: {len(cot_addition)} chars, ~{len(cot_addition)//4} tokens (added automatically when CoT is on)")

 System prompt: 655 chars, ~163 tokens
 CoT addition: 537 chars, ~134 tokens (added automatically when CoT is on)


## 3. Few-Shot Examples

**In-context learning** means teaching a model by showing it examples inside the prompt. No retraining needed.

The tradeoff: each example uses tokens from the context window. The two examples below use about 270 tokens that are then not available for the student question.

Experiment: Toggle few-shot off in the UI. Does the model still follow the four-header format? Small models usually need few-shot more than large models.

In [7]:
# These examples teach the model what good output looks like.
# Change these to match your own assistant's task.
few_shot_examples = [
    # Example 1: TypeError (type mismatch)
    {"role": "user",
     "content": "Student code:\nresult = 'Total: ' + 42\n\n"
               "Error:\nTypeError: can only concatenate str (not \"int\") to str"},
    {"role": "assistant",
     "content": "Diagnosis: Python says it cannot join a string with an integer.\n"
               "Root cause: The + operator needs matching types on both sides.\n"
               "Check: What does type(42) return? What about type('42')? What changes?\n"
               "Review: When mixing text and numbers, one side needs to be converted."},
    # Example 2: Mutable default argument (logic error, no crash)
    {"role": "user",
     "content": "Student code:\ndef register(name, attendees=[]):\n"
               "    attendees.append(name)\n"
               "    return attendees\n\n"
               "print(register('Alice'))\n"
               "print(register('Bob'))  # Returns ['Alice', 'Bob'] not ['Bob']\n\n"
               "No crash, but Bob's list includes Alice."},
    {"role": "assistant",
     "content": "Diagnosis: The list keeps items from previous calls instead of starting fresh.\n"
               "Root cause: Python creates the default list once, then reuses the same object.\n"
               "Check: Call register('X') three times. Does id(attendees) change between calls?\n"
               "Review: Never use a mutable object (list, dict) as a default argument."},
]

print(f" Few-shot examples: {len(str(few_shot_examples))}, chars, ~{len(str(few_shot_examples))//4} tokens") # roughly 

 Few-shot examples: 1063, chars, ~265 tokens


## 4. Test Cases

**Evaluation design**: You cannot improve what you cannot measure.

Seven test cases ordered from easy to hard. All come from real bugs (sources in comments).

Cases 1-5 have clear error messages. Cases 6-7 are **ambiguous**: the error does not directly point to the root cause. These are where CoT *might* help, depending on model size.

Each case has optional context layers:
- `spec`: What the assignment asks.
- `docs`: Library documentation.
- `hint`: A worked example.

**To reuse this notebook for a different assistant**, replace `test_cases` with your own inputs. The format is the same: a name, input text, expected issue, and optional context.

In [8]:
test_cases = {
    "1. TypeError: str + int [Easy]": {
        # Source: stackoverflow.com/questions/1893507 (3M+ views)
        "code": 'age = 25\nmessage = "I am " + age + " years old."\nprint(message)',
        "error": 'TypeError: can only concatenate str (not "int") to str',
        "spec": "Build a greeting string that includes the user's name and age.",
        "docs": "str(x): Convert x to string. f-strings: f'text {var}' embed variables.",
        "hint": "# Two ways to mix strings and numbers:\n"
               "# greeting = 'Score: ' + str(100)\n"
               "# greeting = f'Score: {100}'",
    },
    "2. IndexError: list out of range [Easy]": {
        # Source: stackoverflow.com/questions/1098643
        "code": "fruits = ['apple', 'banana', 'cherry']\n"
               "for i in range(len(fruits)):\n"
               "    print(fruits[i], fruits[i+1])",
        "error": "IndexError: list index out of range",
        "spec": "Print each fruit and the fruit that comes after it.",
        "docs": "len(lst): Number of elements. Last valid index is len(lst)-1.",
        "hint": "# Stop one element early:\n"
               "# for i in range(len(lst) - 1):\n"
               "#     print(lst[i], lst[i+1])",
    },
    "3. Mutable default argument [Medium]": {
        # Source: toptal.com - listed as the #1 Python mistake
        "code": "def add_item(item, shopping_list=[]):\n"
               "    shopping_list.append(item)\n"
               "    return shopping_list\n\n"
               "print(add_item('milk'))\n"
               "print(add_item('bread'))  # Expected ['bread'], got ['milk', 'bread']",
        "error": "No crash, but wrong output: the list accumulates across calls.",
        "spec": "Write a function that creates a new shopping list each time.",
        "docs": "Default values are evaluated ONCE when the function is defined.",
        "hint": "# Safe pattern:\n"
               "# def func(item, lst=None):\n"
               "#     if lst is None:\n"
               "#         lst = []\n"
               "#     lst.append(item)\n"
               "#     return lst",
    },
    "4. pandas KeyError: column name [Medium]": {
        # Source: stackoverflow.com/questions/17431924 (1M+ views)
        "code": "import pandas as pd\n"
               "student_data = pd.DataFrame({'Name': ['Alice', 'Bob'], 'Age': [25, 30]})\n"
               "avg_age = student_data['age'].mean()",
        "error": "KeyError: 'age'",
        "spec": "Calculate the average age from the DataFrame.",
        "docs": "Column access is case-sensitive: df['Age'] != df['age'].",
        "hint": "# Check column names first:\n"
               "# print(student_data.columns.tolist())\n"
               "# Then use exact case: student_data['Age'].mean()",
    },
    "5. UnboundLocalError: variable scope [Hard]": {
        # Source: toptal.com - listed as Python mistake #4
        "code": "count = 0\n\n"
               "def increment():\n"
               "    count += 1\n"
               "    return count\n\n"
               "print(increment())",
        "error": "UnboundLocalError: cannot access local variable 'count'",
        "spec": "Write a function that increments a global counter by 1.",
        "docs": "Assignment inside a function makes the variable local. Use 'global' to modify a global.",
        "hint": "# Two fixes:\n"
               "# 1. global count\n"
               "# 2. def increment(current): return current + 1",
    },
    "6. Two bugs hiding each other [Hard - CoT]": {
        # TWO bugs interact. CoT may help the model trace through both.
        "code": "def average_score(scores):\n"
               "    total = 0\n"
               "    for i in range(1, len(scores)):\n"
               "        total = scores[i]\n"
               "    return total / len(scores)\n\n"
               "result = average_score([80, 90, 70, 100])\n"
               "print('Average:', result)  # Expected 85.0, got 25.0",
        "error": "No crash, but wrong output: expected 85.0, got 25.0",
        "spec": "Return the average of a list of numbers.",
        "docs": "range(1, n) starts at 1, not 0. += adds to a variable. = replaces it.",
        "hint": "# Correct averaging:\n"
               "# total = 0\n"
               "# for i in range(len(scores)):\n"
               "#     total += scores[i]\n"
               "# return total / len(scores)",
    },
    "7. Misleading error line [Hard - CoT]": {
        # Error points to the WRONG line. CoT may help trace the real cause.
        "code": "names = ['alice', 'bob', 'charlie']\n"
               "upper_names = []\n"
               "for name in names:\n"
               "    upper_names.append(name.upper)\n\n"
               "greeting = upper_names[0] + ' is here'\n"
               "print(greeting)",
        "error": "TypeError: can only concatenate str (not 'builtin_function_or_method') to str\n"
                "(Error points to the greeting line, not the append line.)",
        "spec": "Make uppercased names, then print a greeting with the first one.",
        "docs": "str.upper() returns a string. str.upper without () is a method reference, not a string.",
        "hint": "# Method call vs reference:\n"
               "# 'hello'.upper   --> method object (not a string!)\n"
               "# 'hello'.upper() --> 'HELLO'",
    },
    "8. Modifying list while looping [Hard]": {
        # Source: toptal.com - Python mistake #2, SO #1207406 (1M+ views)
        # Deleting items while iterating skips elements silently.
        "code": "numbers = [1, 2, 3, 4, 5, 6]\n"
               "for num in numbers:\n"
               "    if num % 2 == 0:\n"
               "        numbers.remove(num)\n\n"
               "print(numbers)  # Expected [1, 3, 5], got [1, 3, 5, 6]  -- 6 survived!",
        "error": "No crash. Expected [1, 3, 5] but got [1, 3, 5, 6]. The number 6 was not removed.",
        "spec": "Remove all even numbers from the list.",
        "docs": "Removing from a list while iterating shifts indices. The loop may skip the next element.",
        "hint": "# Safe patterns:\n"
               "# 1. Build a new list: [x for x in numbers if x % 2 != 0]\n"
               "# 2. Iterate over a copy: for num in numbers[:]:",
    },
    "9. Off-by-one in factorial [Hard]": {
        # Source: geeksforgeeks.org, rollbar.com, positiwise.com -- appears in multiple tutorials
        # range(1, n) vs range(1, n+1) -- classic silent logic error
        "code": "def factorial(n):\n"
               "    result = 1\n"
               "    for i in range(1, n):\n"
               "        result = result * i\n"
               "    return result\n\n"
               "print(factorial(5))  # Expected 120, got 24",
        "error": "No crash. Expected 120, got 24. The answer is 5! / 5 = 24.",
        "spec": "Calculate the factorial of n (n! = 1 * 2 * 3 * ... * n).",
        "docs": "range(1, n) produces [1, 2, ..., n-1]. It stops BEFORE n.",
        "hint": "# range(1, 5) gives [1, 2, 3, 4] -- no 5!\n"
               "# range(1, 6) gives [1, 2, 3, 4, 5]",
    },
    "10. Silent except hides real bug [Hard]": {
        # Source: dev.to/hackyrupesh (2024), realpython.com
        # bare except + pass makes ALL errors invisible
        "code": "data = {'price': '29.99', 'quantity': 'three'}\n\n"
               "try:\n"
               "    total = float(data['price']) * int(data['quantity'])\n"
               "except:\n"
               "    pass\n\n"
               "print('Order total:', total)  # NameError: total is not defined",
        "error": "NameError: name 'total' is not defined. But there IS a try/except!",
        "spec": "Calculate the order total by converting strings to numbers.",
        "docs": "A bare except catches ALL errors, including ones you did not expect. pass silently does nothing.",
        "hint": "# What error is actually being caught?\n"
               "# Try: except ValueError as e: print(e)\n"
               "# Now you can see what went wrong.",
    },
}

print("Loaded", len(test_cases), "test cases.")

Loaded 10 test cases.


## 5. Inference Engine

This code assembles the prompt, calls the model, and records the result.

**Streaming**: Tokens appear one by one using `yield`. Same work for the model, but feels faster.

**CoT behavior**: When enabled, the chat shows `(thinking...)` while the model reasons, then the final answer streams in. The **Thinking tab** shows the full reasoning afterward. Note: small models often produce broken or incomplete thinking. That is itself an important observation.

**Why functions?** Gradio needs a function to call when you click a button. That is the only reason this section uses `def`.

In [9]:
experiment_log = []

def format_log():
    """Build a readable text block showing all past runs."""
    if len(experiment_log) == 0:
        return "No experiments yet. Click Ask to start."
    log_lines = []
    for r in reversed(experiment_log[-20:]):
        header = "#" + str(r["run"])
        header = header + " | " + r["scenario"]
        header = header + " | " + r["mode"]
        header = header + " | " + r["few_shot"]
        header = header + " | in:" + str(r["in_tok"])
        header = header + " | out:" + str(r["out_tok"])
        header = header + " | " + str(r["latency"]) + "s"
        log_lines.append(header)
        log_lines.append("-" * 60)
        log_lines.append(r["response"][:200])
        log_lines.append("=" * 60 + "\n")
    return "\n".join(log_lines)


def get_visible_text(raw):
    """Hide <thinking> block during streaming, show answer only."""
    if "<thinking>" in raw and "</thinking>" not in raw:
        return "(thinking...)"
    if "<thinking>" in raw and "</thinking>" in raw:
        return re.sub(r"<thinking>.*?</thinking>", "", raw, flags=re.DOTALL).strip()
    return raw


def run_assistant(user_text, system_prompt_text, cot_addition_text, use_few_shot,
              temperature, max_tokens, scenario_name, use_spec, use_docs,
              use_hint, use_cot, model_choice, chat_history):
    """Called every time the student clicks Ask. Streams tokens live."""
    
    # Step 1: Build system prompt (user edits + optional CoT addition)
    active_prompt = system_prompt_text
    if use_cot == True:
        active_prompt = active_prompt + "\n\n" + cot_addition_text

    messages = [{"role": "system", "content": active_prompt}]

    if use_few_shot == True:
        for example in few_shot_examples:
            messages.append(example)

    # Step 2: Build user message from scenario + context layers
    user_message = ""
    if scenario_name != "(none)" and scenario_name in test_cases:
        scenario = test_cases[scenario_name]
        user_message = "Student code:\n" + scenario["code"] + "\n\nError:\n" + scenario["error"] + "\n"
        if use_spec == True and scenario.get("spec"):
            user_message = user_message + "\nAssignment spec:\n" + scenario["spec"] + "\n"
        if use_docs == True and scenario.get("docs"):
            user_message = user_message + "\nDocs reference:\n" + scenario["docs"] + "\n"
        if use_hint == True and scenario.get("hint"):
            user_message = user_message + "\nWorked example:\n" + scenario["hint"] + "\n"
        if user_text.strip():
            user_message = user_message + "\nStudent note: " + user_text.strip()
    else:
        user_message = user_text.strip()
        if not user_message:
            yield chat_history, "", "Type a question or select a scenario.", format_log(), "", ""
            return

    if use_cot == True:
        user_message = user_message + "\n\nRemember: start your response with <thinking>"

    messages.append({"role": "user", "content": user_message})

    # Step 3: White Box
    white_box_text = ""
    for msg in messages:
        white_box_text = white_box_text + "[" + msg["role"].upper() + "]\n"
        white_box_text = white_box_text + msg["content"] + "\n"
        white_box_text = white_box_text + "-" * 40 + "\n\n"

    # Step 4: Count input tokens
    input_tokens = len(model.tokenize(white_box_text.encode("utf-8")))

    effective_max_tokens = int(max_tokens)
    if use_cot == True:
        effective_max_tokens = min(int(max_tokens) + 200, 512)

    # Step 5: Prepare chat history
    if chat_history == None:
        chat_history = []
    display_name = user_text.strip() if user_text.strip() else "[" + scenario_name + "]"
    chat_history.append({"role": "user", "content": display_name})
    chat_history.append({"role": "assistant", "content": ""})

    # Step 6: Stream tokens from whichever model is selected
    is_api = (model_choice in api_models)
    raw_response = ""
    output_tokens = 0
    start_time = time.perf_counter()

    if is_api:
        mode_label = model_choice
        stream = api_client.chat.completions.create(
            model=api_models[model_choice],
            messages=messages,
            max_tokens=effective_max_tokens,
            temperature=float(temperature),
            stream=True,
        )
        for chunk in stream:
            token_text = chunk.choices[0].delta.content or ""
            if token_text:
                raw_response = raw_response + token_text
                output_tokens = output_tokens + 1
                chat_history[-1]["content"] = get_visible_text(raw_response)
                yield chat_history, white_box_text, "", format_log(), "", ""
    else:
        mode_label = "Local"
        stream = model.create_chat_completion(
            messages=messages,
            max_tokens=effective_max_tokens,
            temperature=float(temperature),
            top_p=1.0,
            stream=True,
        )
        for chunk in stream:
            token_text = chunk["choices"][0]["delta"].get("content", "")
            if token_text:
                raw_response = raw_response + token_text
                output_tokens = output_tokens + 1
                chat_history[-1]["content"] = get_visible_text(raw_response)
                yield chat_history, white_box_text, "", format_log(), "", ""

    latency = round(time.perf_counter() - start_time, 2)

    # Step 7: Parse thinking from final response
    thinking_text = ""
    display_answer = raw_response
    if "<thinking>" in raw_response:
        match = re.search(r"<thinking>(.*?)</thinking>", raw_response, re.DOTALL)
        if match:
            thinking_text = match.group(1).strip()
            display_answer = re.sub(r"<thinking>.*?</thinking>", "", raw_response, flags=re.DOTALL).strip()
        else:
            thinking_text = raw_response.replace("<thinking>", "").strip()
            display_answer = "(Model started thinking but did not finish. Try a larger model or turn CoT off.)"

    if use_cot == True:
        mode_label = mode_label + "+CoT"

    # Step 8: Metrics
    pct = (input_tokens * 100) // context_window_size
    metrics = "Model: " + mode_label + "\n"
    metrics = metrics + "Input tokens: " + str(input_tokens) + " / " + str(context_window_size) + " (" + str(pct) + "%)\n"
    metrics = metrics + "Output tokens: " + str(output_tokens) + "\n"
    metrics = metrics + "Prompting: " + ("few-shot" if use_few_shot else "zero-shot") + "\n"
    metrics = metrics + "Latency: " + str(latency) + "s"

    # Step 9: Log
    experiment_log.append({
        "run": len(experiment_log) + 1,
        "scenario": scenario_name[:25],
        "mode": mode_label,
        "few_shot": "few-shot" if use_few_shot else "zero-shot",
        "in_tok": input_tokens,
        "out_tok": output_tokens,
        "latency": latency,
        "response": display_answer,
    })

    # Step 10: Final chat update
    badge = "\n`" + str(latency) + "s | in:" + str(input_tokens) + " | out:" + str(output_tokens) + " | " + mode_label + "`"
    chat_history[-1]["content"] = display_answer + badge

    # Step 11: Thinking tab
    if not use_cot:
        thinking_display = "(CoT is off. Check the CoT box to see the model think step-by-step.)"
    elif not thinking_text:
        thinking_display = "(CoT is on, but the model did not produce <thinking> tags.)"
    else:
        thinking_display = thinking_text

    yield chat_history, white_box_text, metrics, format_log(), "", thinking_display


print("Inference engine ready.")

Inference engine ready.


## 6. Gradio Chatbot UI

### Experiment Guide

| # | Experiment | What to change | What to look for |
|---|-----------|---------------|------------------|
| 1 | **Prompt matters** | Delete "NEVER give the answer" rule | Does the model give away solutions? |
| 2 | **Few-shot effect** | Toggle few-shot off on Scenario 1 | Does the 4-header format hold? |
| 3 | **Context scaling** | Scenario 3: no context → +all | More tokens — better response? |
| 4 | **Temperature** | Same scenario at 0.0, 0.5, 1.0 | More random at high temp? |
| 5 | **Model size** | Same scenario on Local → 8B → 32B → 70B | Quality vs size? |
| 6 | **CoT sweet spot** | Scenario 6 with CoT on each model size | Which size benefits most? |
| 7 | **Two bugs** | Scenario 6 without/with CoT on 32B | Does the model catch BOTH bugs? |
| 8 | **Misleading error** | Scenario 7 without/with CoT on 32B | Does CoT trace to the real bug? |
| 9 | **CoT overkill** | Scenario 1 + CoT on 70B | Wasted tokens on easy + big? |

In [10]:
scenario_choices = ["(none)"]
for name in test_cases:
    scenario_choices.append(name)

model_choices = ["Local"]
for api_name in api_models:
    model_choices.append(api_name)

custom_css = """
    .gradio-container { font-family: sans-serif !important; }
    code, pre, textarea { font-family: monospace !important; }
    .chatbot .message code { background: #f1f5f9 !important; padding: 2px 4px !important; }
    .thinking-box textarea { background: #fffbeb !important; }
"""

app = gr.Blocks(title="Debugging Tutor")

with app:
    gr.Markdown("# Debugging Tutor\nExperiment with every part of the LLM pipeline. Change settings, then click **Ask**.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Debugging Tutor", height=400)
            with gr.Row():
                user_input = gr.Textbox(label="Your message", scale=10, lines=3)
                ask_btn = gr.Button("Ask", variant="primary", scale=1, min_width=80)
            clear_btn = gr.Button("New Chat", variant="secondary")

        with gr.Column(scale=2):
            gr.Markdown("### Controls")
            scenario_dd = gr.Dropdown(choices=scenario_choices, value="(none)", label="Scenario")
            model_dd = gr.Dropdown(choices=model_choices, value="Local", label="Model")

            with gr.Accordion("Prompting Strategy", open=True):
                cot_cb = gr.Checkbox(value=False, label="Enable Chain-of-Thought (CoT)")
                few_shot_cb = gr.Checkbox(value=True, label="Enable few-shot examples")

            with gr.Accordion("Context Layers", open=False):
                spec_cb = gr.Checkbox(value=False, label="spec (assignment description)")
                docs_cb = gr.Checkbox(value=False, label="docs (library reference)")
                hint_cb = gr.Checkbox(value=False, label="hint (worked example)")

            with gr.Accordion("Model Parameters", open=False):
                temp_slider = gr.Slider(minimum=0.0, maximum=1.5, value=0.2, step=0.1, label="Temperature")
                max_tok_slider = gr.Slider(minimum=64, maximum=512, value=192, step=32, label="Max output tokens")

    with gr.Accordion("System Prompt & CoT (click to edit)", open=False):
        with gr.Row():
            system_prompt_box = gr.Textbox(value=system_prompt, label="System Prompt (always used)", lines=6, interactive=True, scale=3)
            cot_addition_box = gr.Textbox(value=cot_addition, label="CoT Addition (appended when CoT is on)", lines=6, interactive=True, scale=2)
        reset_prompt_btn = gr.Button("Reset both to default", variant="secondary", size="sm")

    with gr.Tabs():
        with gr.TabItem("White Box"):
            white_box = gr.Textbox(label="Full prompt sent to model", lines=15, interactive=False)
        with gr.TabItem("Thinking"):
            thinking_box = gr.Textbox(label="Model reasoning (only when CoT is on)", lines=12, interactive=False, elem_classes=["thinking-box"])
        with gr.TabItem("Metrics"):
            metrics_box = gr.Textbox(label="Token usage and latency", lines=6, interactive=False)
        with gr.TabItem("Experiment Log"):
            log_box = gr.Textbox(label="All runs this session", lines=12, interactive=False, value=format_log())

    inputs = [user_input, system_prompt_box, cot_addition_box, few_shot_cb, temp_slider,
              max_tok_slider, scenario_dd, spec_cb, docs_cb, hint_cb, cot_cb, model_dd, chatbot]
    outputs = [chatbot, white_box, metrics_box, log_box, user_input, thinking_box]

    ask_btn.click(fn=run_assistant, inputs=inputs, outputs=outputs)
    user_input.submit(fn=run_assistant, inputs=inputs, outputs=outputs)

    def clear_chat():
        return [], "", "", format_log(), "", ""
    clear_btn.click(fn=clear_chat, outputs=outputs)

    def reset_prompts():
        return system_prompt, cot_addition
    reset_prompt_btn.click(fn=reset_prompts, outputs=[system_prompt_box, cot_addition_box])

app.launch(share=True, inline=True, height=900, theme=gr.themes.Soft(), css=custom_css)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7ca380566ba1a907f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Reflection

After running at least 8-10 experiments, answer these questions:

### Prompt Engineering
1. What system prompt rule had the biggest impact? What happened when you removed it?
2. Did few-shot examples help more with output *format* or *content*?

### Chain-of-Thought and Model Size
3. Run Scenario 6 with CoT OFF on each model size. Which found both bugs? Which only found one?
4. Now run Scenario 6 with CoT ON on each size. Did CoT help smaller models? Did it help the largest?
5. At what model size did CoT stop making a difference? Why do you think that is?
6. Run Scenario 1 (easy) with CoT on the largest model. Useful, or wasted tokens?

### Key Insight
7. Complete this sentence with evidence from your experiments: "CoT helps most when the model is ______ enough to reason but ______ enough to need the help."

### Context Window
8. How did input tokens change as you added context layers? Was there a sweet spot?

### The Bigger Picture
9. For a production debugging tutor, which combination of settings would you pick and why?
10. Pick a different LLM use case and sketch what each component would look like.